In [ ]:
import numpy as np
import pandas as pd
import rasterio as rio
from rasterio.transform import Affine
import os
from tqdm import tqdm
from pathlib import Path

In [ ]:
ROOT = Path.cwd().parent
DATA = ROOT / 'data'
RESULTS = ROOT / 'results'

In [ ]:
habitats_lv2_ = rio.open(RESULTS / 'terrestrial-habitat-class-maps' / 'habitat-classifications-lv2-1km.tif')
country_raster_1km_ = rio.open(RESULTS / 'country-rasters' / 'country-raster-eck4-1km.tif')
DEM_ = rio.open(DATA / 'GMTED2010_EckertIV.tif')
Path(RESULTS / 'area-of-habitat-1km').mkdir(parents=True, exist_ok=True)

# 5 — Area of Habitat (AOH) at 1 km, upscaled to 10 km & 50 km

Merges the former `5-make-AOH`, `5-make-AOH-10km`, `5-make-AOH-50km` into one step.

1. **`make_AOH_for_a_taxon`** — 1 km AOH per species = range × suitable habitat (IUCN lvl-2) × elevation band; writes `area-of-habitat-1km/{taxon}/` + `catalogue-{taxon}.csv`.
2. **`upscale_AOH(taxon, new_res, agg)`** — block-aggregate 1 km AOH to coarser grids:
   - **10 km, `agg='max'`** → presence/absence (uint8), for species-richness counts.
   - **50 km, `agg='sum'`** → habitat area (uint16), for rarity-weighted richness.

> Note: the 1 km DEM is read from a hard-coded external path in the next cell — repoint it if needed.

In [ ]:
def make_AOH_for_a_taxon(taxon):
    Path(RESULTS / 'area-of-habitat-1km' / taxon).mkdir(parents=True, exist_ok=True)
    habitat_preferences = pd.read_csv(RESULTS / 'iucn-assessments' / f'habitat-preferences-{taxon}.csv')
    range_index = pd.read_csv(RESULTS / 'species-range-rasters' / f'range-index-{taxon}.csv')
    elevation_range = pd.read_csv(RESULTS / 'iucn-assessments' / f'elevation-range-{taxon}.csv')
    invalid_index = []
    # range_index['have_a_AOH'] = np.nan

    
    for i in tqdm(range_index.index):
        a_species = range_index.loc[i, 'name']
        a_path = RESULTS / 'species-range-rasters' / taxon / f'{a_species}.tif'
        a_range = rio.open(a_path).read(1) 

        try:
            lower_ele = elevation_range[elevation_range['scientific_name'] == a_species]['lower'].values[0]
        except:
            lower_ele = np.nan 

        try:
            upper_ele = elevation_range[elevation_range['scientific_name'] == a_species]['upper'].values[0]
        except:
            upper_ele = np.nan

        all_codes_ = habitat_preferences[habitat_preferences['scientific_name'] == a_species]['code'].values.tolist()
        all_codes = [int(a.split('_')[0]) * 100 + int(a.split('_')[1]) if '_' in a else int(a) * 100 for a in all_codes_]

        y_1 = range_index.loc[i, 'row_min']
        y_2 = range_index.loc[i, 'row_max']
        x_1 = range_index.loc[i, 'col_min']
        x_2 = range_index.loc[i, 'col_max']
        a_window = ((y_1, y_2+1), (x_1, x_2+1)) 

        local_DEM = DEM_.read(1, window=a_window)
        local_habitats = habitats_lv2_.read(1, window=a_window) 

        habitat_mask = np.isin(local_habitats, all_codes)

        if np.isnan(lower_ele) & ~np.isnan(upper_ele):
            elevation_mask = np.where(local_DEM <= upper_ele, True, False) 
        elif ~np.isnan(lower_ele) & np.isnan(upper_ele):
            elevation_mask = np.where(local_DEM >= lower_ele, True, False) 
        elif np.isnan(lower_ele) & np.isnan(upper_ele):
            elevation_mask = np.where(~np.isnan(local_DEM), True, False)
        else: 
            elevation_mask = np.where((local_DEM >= lower_ele) & (local_DEM <= upper_ele), True, False)

        a_AOH = np.where(habitat_mask & elevation_mask, a_range, 0)

        if np.all(a_AOH == 0):
            invalid_index.append(i)
            range_index.loc[i, 'have_a_AOH'] = False

        else:
            range_index.loc[i, 'have_a_AOH'] = True
            output_path = RESULTS / 'area-of-habitat-1km' / taxon / f'{a_species}.tif'
            new_transform = country_raster_1km_.transform * Affine.translation(x_1, y_1)
            with rio.open(
                    output_path,
                    'w',
                    driver='GTiff',
                    height=a_AOH.shape[0],
                    width=a_AOH.shape[1],
                    count=1,
                    dtype=a_AOH.dtype, 
                    crs=country_raster_1km_.crs,
                    transform=new_transform,
                    compress='lzw',
                ) as dst:
                    dst.write(a_AOH, 1)
    

    range_index.to_csv(RESULTS / 'area-of-habitat-1km' / f'catalogue-{taxon}.csv', index=False)

In [ ]:
def upscale_AOH(taxon, new_res, agg='sum'):
    scale_dict = {'10km': 10, '50km': 50}
    scale = scale_dict[new_res]
    
    country_raster_1_ = rio.open(RESULTS / 'country-rasters' / 'country-raster-eck4-1km.tif')
    country_raster_2_ = rio.open(RESULTS / 'country-rasters' / f'country-raster-eck4-{new_res}.tif')

    a = int(abs(country_raster_1_.transform[5] - country_raster_2_.transform[5]) / 1e3)   # the number of offset grid cells along the row direction, under the original resolution of 1km 
    b = int(abs(country_raster_1_.transform[2] - country_raster_2_.transform[2]) / 1e3)   # the number of offset grid cells along the col direction, under the original resolution of 1km

    catalogue = pd.read_csv(RESULTS / 'area-of-habitat-1km' / f'catalogue-{taxon}.csv')
    Path(RESULTS / f'area-of-habitat-{new_res}' / taxon).mkdir(parents=True, exist_ok=True)
    valid_species = catalogue[catalogue['have_a_AOH'] == True]
    new_catalogue = valid_species.copy()
    new_catalogue['row_min'] = np.nan
    new_catalogue['row_max'] = np.nan
    new_catalogue['col_min'] = np.nan
    new_catalogue['col_max'] = np.nan

    for i in tqdm(valid_species.index):
        row_min_src = catalogue.loc[i, 'row_min']
        row_max_src = catalogue.loc[i, 'row_max']
        col_min_src = catalogue.loc[i, 'col_min']
        col_max_src = catalogue.loc[i, 'col_max']

        row_min_dst = (row_min_src + a) // scale
        row_max_dst = (row_max_src + a) // scale
        col_min_dst = (col_min_src + b) // scale
        col_max_dst = (col_max_src + b) // scale

        # print(row_min_src, row_max_src, col_min_src, col_max_src)
        # print(row_min_dst, row_max_dst, col_min_dst, col_max_dst)

        # print(949*10, 9481 + 9, 950*10, 963*10, 9623+9, 964*10)
        # print(2756*10, 27558 + 8, 2757*10, 2771*10, 27705+8, 2772*10)

        offset_x = col_min_src + b - col_min_dst * scale
        offset_y = row_min_src + a - row_min_dst * scale
        # print(offset_x, offset_y)

        a_species = catalogue.loc[i, 'name']
        a_src = rio.open(RESULTS / 'area-of-habitat-1km' / f'{taxon}' / f'{a_species}.tif')
        a_AOH_src = a_src.read(1)

        H_src = row_max_src - row_min_src + 1
        W_src = col_max_src - col_min_src + 1
        H_dst = row_max_dst - row_min_dst + 1
        W_dst = col_max_dst - col_min_dst + 1
        # a_AOH_dst = np.zeros(shape=(H_dst, W_dst))
        a_AOH_mediate = np.zeros(shape=(H_dst * scale, W_dst * scale))
        # print(a_AOH_dst_to_1km.shape)
        a_AOH_mediate[offset_y:offset_y + H_src, offset_x:offset_x + W_src] = a_AOH_src
        # a_AOH_dst = a_AOH_mediate.reshape(H_dst, scale, W_dst, scale).max(axis=(1, 3)).astype(np.uint8)
        if agg == 'max':
            a_AOH_dst = a_AOH_mediate.reshape(H_dst, scale, W_dst, scale).max(axis=(1, 3)).astype(np.uint8)   # presence/absence
        else:
            a_AOH_dst = a_AOH_mediate.reshape(H_dst, scale, W_dst, scale).sum(axis=(1, 3)).astype(np.uint16)  # habitat area

        transform_10km = country_raster_2_.transform

        x0 = transform_10km.c + col_min_dst * transform_10km.a
        y0 = transform_10km.f + row_min_dst * transform_10km.e

        new_transform = rio.Affine(transform_10km.a, 0, x0,
                                0, transform_10km.e, y0)

        output_path = RESULTS / f'area-of-habitat-{new_res}' / taxon / f'{a_species}.tif'
        with rio.open(
                output_path,
                'w',
                driver='GTiff',
                height=a_AOH_dst.shape[0],
                width=a_AOH_dst.shape[1],
                count=1,
                dtype=a_AOH_dst.dtype, 
                crs=country_raster_2_.crs,
                transform=new_transform,
                compress='lzw',
            ) as dst:
                dst.write(a_AOH_dst, 1)

        new_catalogue.loc[i, 'row_min'] = int(row_min_dst)
        new_catalogue.loc[i, 'row_max'] = int(row_max_dst)
        new_catalogue.loc[i, 'col_min'] = int(col_min_dst)
        new_catalogue.loc[i, 'col_max'] = int(col_max_dst)
        new_catalogue.to_csv(RESULTS / f'area-of-habitat-{new_res}' / f'catalogue-{taxon}.csv', index=False)

## Run

In [ ]:
TAXA = ['amphibians', 'birds', 'mammals', 'reptiles']

# 1) 1 km AOH per species
for taxon in TAXA:
    make_AOH_for_a_taxon(taxon)

In [ ]:
catalogue_birds = pd.read_csv(RESULTS / 'area-of-habitat-1km' / 'catalogue-birds.csv', index_col=0)
red_list_category_birds = pd.read_csv(RESULTS / 'iucn-assessments' / 'redlist-category-birds.csv', index_col=0)
catalogue_birds['category'] = red_list_category_birds['category']
catalogue_birds.to_csv(RESULTS / 'area-of-habitat-1km' / 'catalogue-birds.csv')

In [ ]:
# 2) upscale to 10 km (presence, max) and 50 km (area, sum)
for taxon in TAXA:
    upscale_AOH(taxon, '10km', agg='max')
for taxon in TAXA:
    upscale_AOH(taxon, '50km', agg='sum')